In [1]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

In [14]:
DUCKDB_PATH = "../airbnb.duckdb"  # adjust to your actual path
OUTPUT_PARQUET = "export/price_predictions.parquet"
PRICE_OUTLIER_CAP = 9000  # same cap you applied in Power BI — keep consistent
 
NUMERIC_FEATURES = [
    "accommodates",
    "minimum_nights",
    "occupancy_rate_pct",
]
 
CATEGORICAL_FEATURES = [
    "room_type",
    "neighbourhood",
    "host_is_superhost",
]
 
TARGET = "price"

In [12]:
 
con = duckdb.connect(DUCKDB_PATH)
query1 = f"""SELECT * 
FROM listings
LIMIT 10;"""

query = f"""
        SELECT
        l.id,
        l.price,
        l.room_type,
        l.accommodates,
        l.minimum_nights,
        l.neighbourhood,
        l.host_is_superhost,
        f.occupancy_rate_pct
        FROM listings l
        LEFT JOIN fct_listing_performance f
        ON l.id = f.listing_id"""


df = con.execute(query).df()
con.close()

print(f"Loaded {len(df)} listings")


Loaded 7418 listings


In [13]:
list(df)

['id',
 'price',
 'room_type',
 'accommodates',
 'minimum_nights',
 'neighbourhood',
 'host_is_superhost',
 'occupancy_rate_pct']

In [18]:
before = len(df)
df = df.replace('$', '')
df = df.astype({TARGET: int})
df = df[df[TARGET] < PRICE_OUTLIER_CAP]
print(f"Dropped {before - len(df)} price outliers (>= {PRICE_OUTLIER_CAP})")
 
# Drop rows with missing target or missing critical features
df = df.dropna(subset=[TARGET])
df = df.dropna(subset=CATEGORICAL_FEATURES + NUMERIC_FEATURES, how="all")
 
# Fill remaining numeric gaps with median, categorical gaps with "Unknown"
for col in NUMERIC_FEATURES:
    df[col] = df[col].fillna(df[col].median())
 
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].fillna("Unknown").astype(str)
 
print(f"Final dataset size: {len(df)}")

ValueError: invalid literal for int() with base 10: '$58.00': Error while type casting for column 'price'